In [ ]:
sku_supercedence.filter(F.col('SKUSTATUS') == 'active') \
    .group_by(['UNIQUE FAMILY CODE', 'SKU']) \
    .agg(F.count('*').alias('CNT')) \
    .filter(F.col('CNT') > 1) \
    .show()

In [ ]:
output.group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE_MODEL_FAMILY']) \
    .agg(F.sum('PERCENT_PROPORTION').alias('WEIGHT_SUM')) \
    .filter(F.col('WEIGHT_SUM') != 1.0) \
    .show()

In [ ]:
ecr.group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 
              'UNIQUE FAMILY CODE', 'SKU']) \
   .agg(F.count('*').alias('CNT')) \
   .filter(F.col('CNT') > 1) \
   .show()

In [ ]:
active_skus.group_by(['UNIQUE FAMILY CODE', 'SKU']) \
    .agg(F.count('*').alias('CNT')) \
    .filter(F.col('CNT') > 1) \
    .show()

In [ ]:
forecast_sku.group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 
                        'UNIQUE_FAMILY_CODE', 'SKU']) \
    .agg(F.count('*').alias('CNT')) \
    .filter(F.col('CNT') > 1) \
    .show()

In [ ]:
dealer_family_sku_sales.group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE',
                                   'UNIQUE FAMILY CODE', 'SKU']) \
    .agg(F.count('*').alias('CNT')) \
    .filter(F.col('CNT') > 1) \
    .show()

In [ ]:
disaggregated.group_by(['MONTH_OF_SALE', 'PARENT_DEALER_CODE', 
                         'UNIQUE_FAMILY_CODE', 'SKU']) \
    .agg(F.count('*').alias('CNT')) \
    .filter(F.col('CNT') > 1) \
    .show()

In [ ]:
disaggregated.select('UNIQUE_FAMILY_CODE').distinct().show()

In [ ]:
# ── CONFIG (add these) ───────────────────────────────────────────────
LOOKBACK_MONTHS = 3   # change to 4 if needed
# ────────────────────────────────────────────────────────────────────

# Derive date bounds dynamically from the forecast table
forecast_month_bounds = (
    session.table(TFT_FORECAST_TABLE)
    .select(
        F.min('MONTH_OF_SALE').alias('MIN_FORECAST_MONTH'),
        F.max('MONTH_OF_SALE').alias('MAX_FORECAST_MONTH')
    )
    .collect()[0]
)

earliest_forecast_month = forecast_month_bounds['MIN_FORECAST_MONTH']
latest_forecast_month   = forecast_month_bounds['MAX_FORECAST_MONTH']

# Lookback start = LOOKBACK_MONTHS before the earliest forecast month
lookback_start = F.add_months(F.lit(earliest_forecast_month), -LOOKBACK_MONTHS)

ecr_raw = (
    session.table(ECR_TABLE)
    .filter(F.col('X_CUSTOMER_TYPE').isin(list(CUSTOMER_TYPE)))
    .filter(F.col('CAL_DATE') >= lookback_start)
    .filter(F.col('CAL_DATE') <  F.lit(latest_forecast_month))
    .with_column('CAL_DATE', F.to_date(F.col('CAL_DATE')))
    .with_column(
        'NET_SALES',
        F.col('INVOICED_SALES') + F.col('CANCELLED_SALES') + F.col('RETURNED_SALES')
    )
)